In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark Introduction")
    .master("local[*]")
    .getOrCreate()
)

spark

In [ ]:
# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema= "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [ ]:
# create emp DataFrame 

emp=spark.createDataFrame(data=emp_data, schema=emp_schema)

In [ ]:
# check number of partitions 

emp.rdd.getNumPartitions()

In [ ]:
# show data
emp.show()

In [ ]:

# Select the Emp> 5000

emp_final= emp.where("salary > 5000")

emp_final.show()

In [ ]:
# Write data as CSV output

emp_final.write.format("csv").save("data/output/1/final_data.csv")

In [ ]:
# Schema for emp
from pyspark.sql.functions import  col, expr
emp.schema

emp_filtered= emp.select(col("name"), col("age"), emp.gender)
emp_filtered.show()

In [ ]:
# Basics Transformations

from pyspark.sql.functions import lit, when,col,expr

# Cast age to int, Salary t double
emp_cast=emp.select(expr("name"), expr("cast(age as int) as age"), emp.gender, emp.salary.cast("double")).where("age > 30")
#emp_cast.show()
#print(emp_cast.printSchema())
#print("#########################")

# Add new Column 
emp_add_column= emp.withColumn("Tax", col("salary") * 0.2).withColumn("oneColumn", lit(1)).withColumn("twoColumn", lit(3)).withColumnRenamed("employee_id", "emp_id")
#emp_add_column.show()
#print("#########################")
# Remove Columns
removed_colums=emp_add_column.drop("oneColumn","twoColumn")
#removed_colums.show()

## Add many Columns
columns = {
    "Taxi": col("salary")*0.3,
    "oneColumn":lit(1),
    "twoColumn":lit(3)
}
result_add_columns=emp.withColumns(columns).limit(5)
#result_add_columns.show()

## Strings and Date Format 

from pyspark.sql.functions import to_date,current_date,current_timestamp,when

data_new_gender=emp.withColumn("new_gender", when(
     col("gender")=="Male" , "M").when(col("gender") == "Female","F").otherwise(None)
)
data_new_gender.show()

print("#########################")

data_date= emp.withColumn("new_date",  to_date(col("hire_date"), "yyyy-MM-dd")).withColumn("now",current_date()).withColumn("now_", current_timestamp())
data_date.printSchema()
data_date.show()

In [ ]:
## Data 

emp_data_1 = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"]
]

emp_data_2 = [
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"


## Create emp DataFrme

emp_data_1= spark.createDataFrame(data=emp_data_1, schema=emp_schema)
emp_data_2= spark.createDataFrame(data=emp_data_2, schema=emp_schema)

## Show Data
emp_data_1.show()
emp_data_2.show()

## Print Schema
emp_data_1.printSchema()
emp_data_2.printSchema()

## union and  unionAll

emp_data= emp_data_1.unionAll(emp_data_2)
emp_data.show()

## OrderBy

from pyspark.sql.functions import desc, asc, col

print("##################################################")
emp_sorted= emp_data.orderBy(col("salary").asc())
emp_sorted.show()

In [ ]:
## Aggregation

from pyspark.sql.functions import  count, col,sum,expr

emp_data_count= emp_sorted.groupBy("department_id").agg(count("employee_id").alias("total_dept_count"))

emp_data_count.show()

print(" #################### ")

emp_data_sum=emp_sorted.groupBy("department_id").agg(sum("salary").alias("total_dept_salary"))
emp_data_sum.show()

print(" #################### ")

emp_data_sum_gender=emp_sorted.groupBy("gender").agg(sum("salary").alias("total_dept_salary"))
emp_data_sum_gender.show()

print(" #################### ")


emp_data_sum_having=emp_sorted.groupBy("department_id").agg(sum("salary").alias("total_dept_salary"))
result=emp_data_sum_having.select("department_id", expr("cast(total_dept_salary as int) total") ).where("total>100000")
result.show()


In [ ]:
## distinct

data=spark.createDataFrame(schema=emp_schema, data=emp_data)
data.show()

print(" ##################  ")

data_unique= data.distinct()

## department_id
data_dp=data.select('department_id').distinct()
data_dp.show()

In [ ]:
## Windows Functions
## Because window functions are not filters
## They are calculations per row:  keeps all rows   adds info
## Window groups rows logically, calculates max per group, but does not remove non-max rows


# groupBy : removes rows, only gives summary

from pyspark.sql.window import  Window
from pyspark.sql.functions import  max, col, desc, row_number,expr

windows_spec = Window.partitionBy(col("department_id"))
max_func= max(col("salary")).over(windows_spec)

data_1=data.withColumn("max_salary",max_func)
data_1.show()


print(" ##############     ################# ")

windows_spec_2=Window.partitionBy(col("department_id")).orderBy(col("salary") .desc())
rn= row_number().over(windows_spec_2)

data_2=data.withColumn("rn",rn).where("rn=2")
data_2.show()

print(" ##############     ################# ")

data_3= data.withColumn("rn",expr("row_number() over(partition by department_id order by salary desc)")).where("rn=2")
data_3.show()

In [ ]:
## Data

# Emp Data & Schema

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

dept_data = [
    ["101", "Sales", "NYC", "US", "1000000"],
    ["102", "Marketing", "LA", "US", "900000"],
    ["103", "Finance", "London", "UK", "1200000"],
    ["104", "Engineering", "Beijing", "China", "1500000"],
    ["105", "Human Resources", "Tokyo", "Japan", "800000"],
    ["106", "Research and Development", "Perth", "Australia", "1100000"],
    ["107", "Customer Service", "Sydney", "Australia", "950000"]
]

dept_schema = "department_id string, department_name string, city string, country string, budget string" 


# Create Dataframe 

data=spark.createDataFrame(data=emp_data, schema=emp_schema)
data_department=spark.createDataFrame(data=dept_data, schema=dept_schema)

data.show()
data_department.show()

print(data.rdd.getNumPartitions())

# Repartiton of data 
emp_partitioned= data.repartition(4, "department_id")
emp_partitioned.rdd.getNumPartitions()


# in Apache Spark:

# 1 partition = 1 task
# 1 task = 1 CPU core

from pyspark.sql.functions import  spark_partition_id 

data_1=data.repartition(4, "department_id").withColumn("partition_num", spark_partition_id())
data_1.show()

In [ ]:
## Inner Join

data_joined = data.alias("e").join(data_department.alias("d"), how='inner', on=data.department_id==data_department.department_id)

data_joined.select("e.employee_id", "e.department_id","e.name", "d.department_name", "d.city").show()